# Phase 5: Targeted Analysis 13: Activation Patching (Interchange Interventions)

## Overview

All prior analyses measure circuit behaviour via ablation (removing components).
This notebook uses **interchange interventions**: swapping activations from one
frequency band's forward pass into another's, to test whether band-specific
information is **causally localized** at specific layers, components, or positions.

## Hypotheses

- **H1**: If phantom specialization holds, patching activations from any band into
  any other band's forward pass should cause the model to predict the source's
  target at high rates (model doesn't functionally differentiate bands).
- **H2**: IIA should be uniformly high across all band pairs (no pair-specific
  asymmetry), consistent with generic transfer.
- **H3**: Patching at repetition positions (16-20) and the prediction position
  should have the strongest causal effect.

## Notebook Structure

1. Setup & Imports
2. Interchange Pair Construction
3. TransformerLens Patching Hooks
4. Core Metric Computation
5. Main Sweep: Residual Stream Across All Layers
6. Component Decomposition at Top Layers
7. Position Sweep at Peak Layer
8. Full 5x5 IIA Matrix at Peak Layer
9. Robustness: Draws 2 & 3
10. Save Results
11-14. Visualizations
15. Cross-Reference with NB05 Edge Importance
16. Summary

## Data Sources

- Test data: `LSC_data/datasets/matched/{draw}/{band}/test.json` (225 examples each)
- Edge importance: `outputs/analysis/` from NB05
- Metrics: Logit difference (primary), IIA (secondary)

In [1]:
import os
import sys
import json
import gc
from pathlib import Path
from dataclasses import dataclass
from typing import List, Dict, Tuple, Optional

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import numpy as np
import pandas as pd
import torch as t
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# --- Paths ---
ISC_ROOT = Path(os.environ.get("PROJECT_ROOT", ".")).resolve()
LSC_DIR = ISC_ROOT / "LSC_circuits"
AUTOCIRCUIT_PATH = os.environ.get("AUTOCIRCUIT_PATH") or str(
    ISC_ROOT / "circuit_discovery" / "auto-circuit"
)
sys.path.insert(0, AUTOCIRCUIT_PATH)
sys.path.insert(0, str(LSC_DIR))

from lsc_acdc_circuit import (
    load_model,
    model_safe_name,
    get_batch_size,
    set_all_seeds,
    cleanup_gpu,
    safe_delete_model,
)

# --- Constants ---
MODELS = ["pythia-70m", "pythia-160m", "pythia-410m", "pythia-1b", "pythia-1.4b"]
BANDS = ["low", "medium", "high", "very_high", "control"]
DRAWS = ["draw_1", "draw_2", "draw_3"]
EVAL_SEED = 123
VARIANT = "matched"
MODEL_LAYERS = {
    "pythia-70m": 6,
    "pythia-160m": 12,
    "pythia-410m": 24,
    "pythia-1b": 16,
    "pythia-1.4b": 24,
}
MODEL_HIDDEN = {
    "pythia-70m": 512,
    "pythia-160m": 768,
    "pythia-410m": 1024,
    "pythia-1b": 2048,
    "pythia-1.4b": 2048,
}
MODEL_HEADS = {
    "pythia-70m": 8,
    "pythia-160m": 12,
    "pythia-410m": 16,
    "pythia-1b": 8,
    "pythia-1.4b": 16,
}

BASE = ISC_ROOT
DATA_DIR = BASE / "LSC_data"
ANALYSIS_DIR = Path("outputs/analysis")
VIZ_DIR = Path("outputs/viz")
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
VIZ_DIR.mkdir(parents=True, exist_ok=True)

# Representative band pairs for main sweep
BAND_PAIRS = [
    ("low", "high"),  # large frequency gap, forward
    ("high", "low"),  # large frequency gap, reverse
    ("low", "control"),  # test vs control
    ("very_high", "medium"),  # intermediate gap
]

N_PAIRS = 100  # interchange pairs per condition

device = "cuda:0"
print(f"Device: {device}")
print(f"CUDA available: {t.cuda.is_available()}")
if t.cuda.is_available():
    print(f"GPU: {t.cuda.get_device_name(0)}")

Device: cuda:0
CUDA available: True
GPU: NVIDIA A100 80GB PCIe


## 2. Interchange Pair Construction

In [2]:
@dataclass
class InterchangePair:
    """A base-source pair for interchange intervention."""

    base_ids: t.Tensor  # (1, seq_len) with BOS prepended
    source_ids: t.Tensor  # (1, seq_len) with BOS prepended
    base_target_id: int  # token ID the base input should predict
    source_target_id: int  # token ID the source input should predict


def load_test_examples(band: str, draw: str) -> List[dict]:
    """Load test examples from dataset JSON."""
    path = DATA_DIR / "datasets" / VARIANT / draw / band / "test.json"
    with open(path) as f:
        data = json.load(f)
    return data["examples"]


def create_interchange_pairs(
    base_band: str,
    source_band: str,
    draw: str,
    n_pairs: int = N_PAIRS,
    bos_id: int = 0,
    seed: int = EVAL_SEED,
) -> List[InterchangePair]:
    """Create interchange pairs from two bands.

    Pairs examples by index (both have 225 examples, same length=21).
    Prepends BOS to get length 22.
    """
    rng = np.random.default_rng(seed)
    base_examples = load_test_examples(base_band, draw)
    source_examples = load_test_examples(source_band, draw)

    n_avail = min(len(base_examples), len(source_examples))
    indices = rng.permutation(n_avail)[:n_pairs]

    pairs = []
    for idx in indices:
        base_ex = base_examples[idx]
        source_ex = source_examples[idx]

        # Prepend BOS
        base_ids = t.tensor([[bos_id] + base_ex["token_ids"]], dtype=t.long)
        source_ids = t.tensor([[bos_id] + source_ex["token_ids"]], dtype=t.long)

        pairs.append(
            InterchangePair(
                base_ids=base_ids,
                source_ids=source_ids,
                base_target_id=base_ex["target_token_id"],
                source_target_id=source_ex["target_token_id"],
            )
        )
    return pairs


def batch_interchange_pairs(
    pairs: List[InterchangePair],
) -> Tuple[t.Tensor, t.Tensor, t.Tensor, t.Tensor]:
    """Stack pairs into batched tensors for efficient processing."""
    base_ids = t.cat([p.base_ids for p in pairs], dim=0)  # (N, seq_len)
    source_ids = t.cat([p.source_ids for p in pairs], dim=0)  # (N, seq_len)
    base_targets = t.tensor([p.base_target_id for p in pairs])  # (N,)
    source_targets = t.tensor([p.source_target_id for p in pairs])  # (N,)
    return base_ids, source_ids, base_targets, source_targets


# Quick test
test_pairs = create_interchange_pairs("low", "high", "draw_1", n_pairs=5, bos_id=0)
print(f"Created {len(test_pairs)} pairs")
print(
    f"Base shape: {test_pairs[0].base_ids.shape}, Source shape: {test_pairs[0].source_ids.shape}"
)
print(
    f"Base target: {test_pairs[0].base_target_id}, Source target: {test_pairs[0].source_target_id}"
)
b, s, bt, st = batch_interchange_pairs(test_pairs)
print(f"Batched: base={b.shape}, source={s.shape}, base_tgt={bt.shape}")

Created 5 pairs
Base shape: torch.Size([1, 22]), Source shape: torch.Size([1, 22])
Base target: 38669, Source target: 5420
Batched: base=torch.Size([5, 22]), source=torch.Size([5, 22]), base_tgt=torch.Size([5])


## 3. TransformerLens Patching Hooks

In [3]:
def make_patch_hook(source_cache: t.Tensor, position: int):
    """Create a hook that patches activations at a specific position.

    Works for hook_resid_post and hook_mlp_out (shape: batch, seq, d_model).
    """

    def hook_fn(activation, hook):
        patched = activation.clone()
        patched[:, position, :] = source_cache[:, position, :]
        return patched

    return hook_fn


def make_attn_patch_hook(
    source_cache: t.Tensor, position: int, head_idx: Optional[int] = None
):
    """Create a hook that patches attention output (hook_z) at a specific position.

    hook_z shape: (batch, seq, n_heads, d_head)
    If head_idx is None, patches all heads.
    """

    def hook_fn(activation, hook):
        patched = activation.clone()
        if head_idx is not None:
            patched[:, position, head_idx, :] = source_cache[:, position, head_idx, :]
        else:
            patched[:, position, :, :] = source_cache[:, position, :, :]
        return patched

    return hook_fn


print("Patching hooks defined.")

Patching hooks defined.


## 4. Core Metric Computation

In [4]:
@t.no_grad()
def compute_interchange_metrics(
    model,
    pairs: List[InterchangePair],
    hook_name: str,
    position: int,
    batch_size: int = 50,
) -> Dict:
    """Run interchange interventions and compute IIA + logit difference.

    For each pair:
    1. Cache source activations at hook_name
    2. Run base input with patched source activations
    3. Check if prediction matches source target (IIA)
    4. Compute logit difference: logit[source_target] - logit[base_target]

    Returns dict with iia, mean_logit_diff, std_logit_diff, n_pairs.
    """
    dev = next(model.parameters()).device
    base_ids, source_ids, base_tgts, source_tgts = batch_interchange_pairs(pairs)
    base_ids = base_ids.to(dev)
    source_ids = source_ids.to(dev)

    n = len(pairs)
    all_iia = []
    all_logit_diff = []

    is_attn = "hook_z" in hook_name

    for start in range(0, n, batch_size):
        end = min(start + batch_size, n)
        b_ids = base_ids[start:end]
        s_ids = source_ids[start:end]
        b_tgts = base_tgts[start:end]
        s_tgts = source_tgts[start:end]

        # 1. Cache source activations
        _, cache = model.run_with_cache(
            s_ids, prepend_bos=False, names_filter=[hook_name]
        )
        source_act = cache[hook_name]  # (batch, seq, ...)
        del cache

        # 2. Run base with patched activations
        if is_attn:
            hook_fn = make_attn_patch_hook(source_act, position)
        else:
            hook_fn = make_patch_hook(source_act, position)

        patched_logits = model.run_with_hooks(
            b_ids, prepend_bos=False, fwd_hooks=[(hook_name, hook_fn)]
        )  # (batch, seq, vocab)

        last_logits = patched_logits[:, -1, :]  # (batch, vocab)
        del source_act, patched_logits

        # 3. IIA: does prediction match source target?
        pred_ids = last_logits.argmax(dim=-1).cpu()
        matches = (pred_ids == s_tgts).float()
        all_iia.append(matches)

        # 4. Logit difference: logit[source_target] - logit[base_target]
        source_logits = last_logits[range(len(s_tgts)), s_tgts.to(dev)].cpu().float()
        base_logits = last_logits[range(len(b_tgts)), b_tgts.to(dev)].cpu().float()
        logit_diffs = source_logits - base_logits
        all_logit_diff.append(logit_diffs)

        del last_logits

    iia_arr = t.cat(all_iia)
    ld_arr = t.cat(all_logit_diff)

    return {
        "iia": iia_arr.mean().item(),
        "mean_logit_diff": ld_arr.mean().item(),
        "std_logit_diff": ld_arr.std().item(),
        "n_pairs": n,
    }


print("Core metric computation defined.")

Core metric computation defined.


## 5. Main Sweep: Residual Stream Across All Layers

For each model (draw_1), run 4 representative band pairs x all layers,
patching `hook_resid_post` at the last position (prediction).

In [5]:
resid_results = []

for model_name in MODELS:
    n_layers = MODEL_LAYERS[model_name]
    print(f"\n{'=' * 60}")
    print(f"Model: {model_name} ({n_layers} layers)")
    print(f"{'=' * 60}")

    model = load_model(model_name, device)
    bos_id = model.tokenizer.bos_token_id
    seq_len = 22  # 21 tokens + BOS
    pred_pos = seq_len - 1  # last position (index 21)

    for base_band, source_band in BAND_PAIRS:
        pairs = create_interchange_pairs(
            base_band,
            source_band,
            "draw_1",
            n_pairs=N_PAIRS,
            bos_id=bos_id,
        )

        for layer in range(n_layers):
            hook_name = f"blocks.{layer}.hook_resid_post"
            metrics = compute_interchange_metrics(
                model,
                pairs,
                hook_name,
                pred_pos,
            )

            resid_results.append(
                {
                    "model": model_name,
                    "base_band": base_band,
                    "source_band": source_band,
                    "layer": layer,
                    "component": "resid",
                    "position": pred_pos,
                    "iia": metrics["iia"],
                    "mean_logit_diff": metrics["mean_logit_diff"],
                    "std_logit_diff": metrics["std_logit_diff"],
                }
            )

        pair_label = f"{base_band}->{source_band}"
        best_layer = max(
            range(n_layers),
            key=lambda l: [
                r["iia"]
                for r in resid_results
                if r["model"] == model_name
                and r["base_band"] == base_band
                and r["source_band"] == source_band
                and r["layer"] == l
            ][0],
        )
        best_iia = [
            r["iia"]
            for r in resid_results
            if r["model"] == model_name
            and r["base_band"] == base_band
            and r["source_band"] == source_band
            and r["layer"] == best_layer
        ][0]
        print(f"  {pair_label}: best layer={best_layer}, IIA={best_iia:.3f}")

    safe_delete_model(model)
    cleanup_gpu()

df_resid = pd.DataFrame(resid_results)
print(f"\nResidual sweep: {len(df_resid)} rows")
df_resid.head()


Model: pythia-70m (6 layers)


'torch_dtype' is deprecated! Use 'dtype' instead!


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Loaded pretrained model pythia-70m into HookedTransformer


  low->high: best layer=5, IIA=0.450


...

  high->low: best layer=17, IIA=0.970


  low->control: best layer=22, IIA=0.990


  very_high->medium: best layer=19, IIA=0.980
Moving model to device:  cpu



Residual sweep: 328 rows


,model,base_band,source_band,layer,component,position,iia,mean_logit_diff,std_logit_diff
0,pythia-70m,low,high,0,resid,21,0.00,0.978234,2.194734
1,pythia-70m,low,high,1,resid,21,0.00,1.045536,2.256275
2,pythia-70m,low,high,2,resid,21,0.00,1.248155,2.343678
3,pythia-70m,low,high,3,resid,21,0.34,8.386784,4.359509
4,pythia-70m,low,high,4,resid,21,0.43,9.526932,4.341701


## 6. Component Decomposition at Top Layers

At the top-3 layers (by mean IIA across band pairs), decompose into
residual, attention, and MLP.

In [6]:
# Identify top-3 layers per model
peak_layers = {}
for model_name in MODELS:
    df_m = df_resid[df_resid["model"] == model_name]
    layer_iia = df_m.groupby("layer")["iia"].mean().sort_values(ascending=False)
    top3 = layer_iia.head(3).index.tolist()
    peak_layers[model_name] = top3
    print(f"{model_name}: top-3 layers = {top3} (IIA = {layer_iia.iloc[:3].values})")

# Component decomposition
component_results = []

for model_name in MODELS:
    n_layers = MODEL_LAYERS[model_name]
    print(f"\n{model_name}:")

    model = load_model(model_name, device)
    bos_id = model.tokenizer.bos_token_id
    seq_len = 22
    pred_pos = seq_len - 1

    for base_band, source_band in BAND_PAIRS:
        pairs = create_interchange_pairs(
            base_band,
            source_band,
            "draw_1",
            n_pairs=N_PAIRS,
            bos_id=bos_id,
        )

        for layer in peak_layers[model_name]:
            for component, hook_name in [
                ("resid", f"blocks.{layer}.hook_resid_post"),
                ("attn", f"blocks.{layer}.attn.hook_z"),
                ("mlp", f"blocks.{layer}.hook_mlp_out"),
            ]:
                metrics = compute_interchange_metrics(
                    model,
                    pairs,
                    hook_name,
                    pred_pos,
                )
                component_results.append(
                    {
                        "model": model_name,
                        "base_band": base_band,
                        "source_band": source_band,
                        "layer": layer,
                        "component": component,
                        "iia": metrics["iia"],
                        "mean_logit_diff": metrics["mean_logit_diff"],
                    }
                )

        print(f"  {base_band}->{source_band}: done")

    safe_delete_model(model)
    cleanup_gpu()

df_comp = pd.DataFrame(component_results)
print(f"\nComponent decomposition: {len(df_comp)} rows")

pythia-70m: top-3 layers = [5, 4, 3] (IIA = [0.4275     0.40750001 0.31      ])
pythia-160m: top-3 layers = [11, 10, 9] (IIA = [0.9575 0.955  0.9325])
pythia-410m: top-3 layers = [23, 22, 21] (IIA = [0.985      0.98       0.97750001])
pythia-1b: top-3 layers = [13, 14, 15] (IIA = [0.99000001 0.99000001 0.98750001])
pythia-1.4b: top-3 layers = [21, 19, 20] (IIA = [0.97250001 0.97250001 0.96750002])

pythia-70m:


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Loaded pretrained model pythia-70m into HookedTransformer


  low->high: done


  high->low: done


...

  low->high: done


  high->low: done


  low->control: done


  very_high->medium: done
Moving model to device:  cpu



Component decomposition: 180 rows


## 7. Position Sweep at Peak Layer

At the overall peak layer per model, sweep patching position across:
- Source tokens (positions 1-5 with BOS offset)
- Target token (position 6)
- Repetition tokens (positions 17-21)
- Prediction position (21, same as last repetition)

In [7]:
# Positions to sweep (with BOS offset: raw pos + 1)
# Raw: source=0-4, target=5, distract=6-15, repeat=16-20
# With BOS: source=1-5, target=6, distract=7-16, repeat=17-21
SWEEP_POSITIONS = {
    "S1": 1,
    "S2": 2,
    "S3": 3,
    "S4": 4,
    "S5": 5,
    "T": 6,
    "D1": 7,
    "D5": 11,
    "D10": 16,  # sample distractors
    "R1": 17,
    "R2": 18,
    "R3": 19,
    "R4": 20,
    "R5_pred": 21,
}

position_results = []

for model_name in MODELS:
    # Use overall peak layer (top-1)
    peak_layer = peak_layers[model_name][0]
    hook_name = f"blocks.{peak_layer}.hook_resid_post"
    print(f"\n{model_name}: position sweep at layer {peak_layer}")

    model = load_model(model_name, device)
    bos_id = model.tokenizer.bos_token_id

    # Use low->high pair (largest gap)
    pairs = create_interchange_pairs(
        "low",
        "high",
        "draw_1",
        n_pairs=N_PAIRS,
        bos_id=bos_id,
    )

    for pos_name, pos_idx in SWEEP_POSITIONS.items():
        metrics = compute_interchange_metrics(
            model,
            pairs,
            hook_name,
            pos_idx,
        )
        position_results.append(
            {
                "model": model_name,
                "peak_layer": peak_layer,
                "position_name": pos_name,
                "position_idx": pos_idx,
                "iia": metrics["iia"],
                "mean_logit_diff": metrics["mean_logit_diff"],
            }
        )
        print(
            f"  pos={pos_name} ({pos_idx}): IIA={metrics['iia']:.3f}, LD={metrics['mean_logit_diff']:.2f}"
        )

    safe_delete_model(model)
    cleanup_gpu()

df_pos = pd.DataFrame(position_results)
print(f"\nPosition sweep: {len(df_pos)} rows")


pythia-70m: position sweep at layer 5


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Loaded pretrained model pythia-70m into HookedTransformer
  pos=S1 (1): IIA=0.000, LD=-6.21


  pos=S2 (2): IIA=0.000, LD=-6.21
  pos=S3 (3): IIA=0.000, LD=-6.21
  pos=S4 (4): IIA=0.000, LD=-6.21


  pos=S5 (5): IIA=0.000, LD=-6.21
  pos=T (6): IIA=0.000, LD=-6.21
  pos=D1 (7): IIA=0.000, LD=-6.21


...

  pos=R2 (18): IIA=0.000, LD=-11.39


  pos=R3 (19): IIA=0.000, LD=-11.34


  pos=R4 (20): IIA=0.000, LD=-11.28


  pos=R5_pred (21): IIA=0.970, LD=12.39
Moving model to device:  cpu



Position sweep: 70 rows


## 8. Full 5x5 IIA Matrix at Peak Layer

All 25 band pairs (including same-band) at the peak layer.

In [8]:
matrix_results = []

for model_name in MODELS:
    peak_layer = peak_layers[model_name][0]
    hook_name = f"blocks.{peak_layer}.hook_resid_post"
    print(f"\n{model_name}: 5x5 matrix at layer {peak_layer}")

    model = load_model(model_name, device)
    bos_id = model.tokenizer.bos_token_id
    pred_pos = 21

    for base_band in BANDS:
        for source_band in BANDS:
            pairs = create_interchange_pairs(
                base_band,
                source_band,
                "draw_1",
                n_pairs=N_PAIRS,
                bos_id=bos_id,
            )
            metrics = compute_interchange_metrics(
                model,
                pairs,
                hook_name,
                pred_pos,
            )
            matrix_results.append(
                {
                    "model": model_name,
                    "peak_layer": peak_layer,
                    "base_band": base_band,
                    "source_band": source_band,
                    "iia": metrics["iia"],
                    "mean_logit_diff": metrics["mean_logit_diff"],
                }
            )
        print(f"  base={base_band}: done")

    safe_delete_model(model)
    cleanup_gpu()

df_matrix = pd.DataFrame(matrix_results)
print(f"\nFull matrix: {len(df_matrix)} rows")


pythia-70m: 5x5 matrix at layer 5


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Loaded pretrained model pythia-70m into HookedTransformer


  base=low: done


  base=medium: done


...

  base=medium: done


  base=high: done


  base=very_high: done


  base=control: done
Moving model to device:  cpu



Full matrix: 125 rows


## 9. Robustness: Draws 2 & 3

Repeat residual sweep at the peak layer for draws 2 and 3.

In [9]:
draw_results = []

for model_name in MODELS:
    peak_layer = peak_layers[model_name][0]
    hook_name = f"blocks.{peak_layer}.hook_resid_post"
    print(f"\n{model_name}: robustness check at layer {peak_layer}")

    model = load_model(model_name, device)
    bos_id = model.tokenizer.bos_token_id
    pred_pos = 21

    for draw in DRAWS:
        for base_band, source_band in BAND_PAIRS:
            pairs = create_interchange_pairs(
                base_band,
                source_band,
                draw,
                n_pairs=N_PAIRS,
                bos_id=bos_id,
            )
            metrics = compute_interchange_metrics(
                model,
                pairs,
                hook_name,
                pred_pos,
            )
            draw_results.append(
                {
                    "model": model_name,
                    "draw": draw,
                    "base_band": base_band,
                    "source_band": source_band,
                    "peak_layer": peak_layer,
                    "iia": metrics["iia"],
                    "mean_logit_diff": metrics["mean_logit_diff"],
                }
            )
        print(f"  {draw}: done")

    safe_delete_model(model)
    cleanup_gpu()

df_draws = pd.DataFrame(draw_results)

# Summary: mean +/- std across draws
draw_summary = (
    df_draws.groupby(["model", "base_band", "source_band"])
    .agg(
        iia_mean=("iia", "mean"),
        iia_std=("iia", "std"),
        ld_mean=("mean_logit_diff", "mean"),
        ld_std=("mean_logit_diff", "std"),
    )
    .reset_index()
)
print(f"\nDraw robustness: {len(df_draws)} rows")
print(draw_summary.to_string(index=False))


pythia-70m: robustness check at layer 5


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Loaded pretrained model pythia-70m into HookedTransformer


  draw_1: done


  draw_2: done


...

Loaded pretrained model pythia-1.4b into HookedTransformer


  draw_1: done


  draw_2: done


  draw_3: done
Moving model to device:  cpu



Draw robustness: 60 rows
      model base_band source_band  iia_mean  iia_std   ld_mean   ld_std
pythia-1.4b      high         low  0.960000 0.010000 10.971561 0.172349
pythia-1.4b       low     control  0.983333 0.005773 12.493924 0.063724
pythia-1.4b       low        high  0.986667 0.015275 12.485341 0.083399
pythia-1.4b very_high      medium  0.973333 0.020817  9.527464 0.197219
pythia-160m      high         low  0.916667 0.020817 10.906489 0.406252
pythia-160m       low     control  0.970000 0.026458 12.821287 0.231825
pythia-160m       low        high  0.966667 0.015275 12.822914 0.111394
pythia-160m very_high      medium  0.940000 0.000000 10.126395 0.030519
  pythia-1b      high         low  0.966667 0.005774 11.033900 0.465725
  pythia-1b       low     control  0.990000 0.010000 12.076318 0.322706
  pythia-1b       low        high  0.996667 0.005773 12.008404 0.075933
  pythia-1b very_high      medium  0.983333 0.020817  9.584230 0.321189
pythia-410m      high         low  0.9

## 10. Save Results

In [10]:
df_resid.to_csv(ANALYSIS_DIR / "activation_patching_resid_sweep.csv", index=False)
df_comp.to_csv(ANALYSIS_DIR / "activation_patching_component_decomp.csv", index=False)
df_pos.to_csv(ANALYSIS_DIR / "activation_patching_position_sweep.csv", index=False)
df_matrix.to_csv(ANALYSIS_DIR / "activation_patching_full_iia_matrix.csv", index=False)
df_draws.to_csv(ANALYSIS_DIR / "activation_patching_draw_robustness.csv", index=False)

# Also save peak layers for NB14
import json as json_mod

with open(ANALYSIS_DIR / "activation_patching_peak_layers.json", "w") as f:
    json_mod.dump(peak_layers, f, indent=2)

print("Saved:")
for fname in [
    "activation_patching_resid_sweep.csv",
    "activation_patching_component_decomp.csv",
    "activation_patching_position_sweep.csv",
    "activation_patching_full_iia_matrix.csv",
    "activation_patching_draw_robustness.csv",
    "activation_patching_peak_layers.json",
]:
    print(f"  {fname}")

Saved:
  activation_patching_resid_sweep.csv
  activation_patching_component_decomp.csv
  activation_patching_position_sweep.csv
  activation_patching_full_iia_matrix.csv
  activation_patching_draw_robustness.csv
  activation_patching_peak_layers.json


## 11. Viz 1: Layer-wise Logit Difference Trajectories

In [11]:
fig, axes = plt.subplots(1, 5, figsize=(22, 4), sharey=True)

pair_colors = {
    ("low", "high"): "#E24A33",
    ("high", "low"): "#348ABD",
    ("low", "control"): "#988ED5",
    ("very_high", "medium"): "#FBC15E",
}

for col, model_name in enumerate(MODELS):
    ax = axes[col]
    df_m = df_resid[df_resid["model"] == model_name]

    for (bb, sb), color in pair_colors.items():
        df_pair = df_m[(df_m["base_band"] == bb) & (df_m["source_band"] == sb)]
        df_pair = df_pair.sort_values("layer")
        label = f"{bb}->{sb}"
        ax.plot(
            df_pair["layer"],
            df_pair["mean_logit_diff"],
            color=color,
            linewidth=1.5,
            label=label,
            marker="o",
            markersize=3,
        )

    ax.set_title(model_name.replace("pythia-", ""), fontsize=11)
    ax.set_xlabel("Layer")
    if col == 0:
        ax.set_ylabel("Mean Logit Difference")
    ax.axhline(y=0, color="gray", linestyle="--", alpha=0.5)
    if col == 4:
        ax.legend(fontsize=7, loc="upper left")

plt.suptitle(
    "Interchange Intervention: Logit Difference by Layer (Residual Stream)", fontsize=12
)
plt.tight_layout()
plt.savefig(VIZ_DIR / "T13_01_layer_logit_diff.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved T13_01_layer_logit_diff.png")

Saved T13_01_layer_logit_diff.png


## 12. Viz 2: Component Decomposition

In [12]:
fig, axes = plt.subplots(1, 5, figsize=(22, 5), sharey=True)

comp_colors = {"resid": "#4C72B0", "attn": "#DD8452", "mlp": "#55A868"}

for col, model_name in enumerate(MODELS):
    ax = axes[col]
    df_m = df_comp[df_comp["model"] == model_name]

    # Average across band pairs per (layer, component)
    df_agg = df_m.groupby(["layer", "component"])["iia"].mean().reset_index()

    layers = sorted(df_agg["layer"].unique())
    x = np.arange(len(layers))
    width = 0.25

    for i, comp in enumerate(["resid", "attn", "mlp"]):
        vals = [
            df_agg[(df_agg["layer"] == l) & (df_agg["component"] == comp)]["iia"].values
            for l in layers
        ]
        vals = [v[0] if len(v) > 0 else 0 for v in vals]
        ax.bar(
            x + (i - 1) * width,
            vals,
            width,
            label=comp,
            color=comp_colors[comp],
            alpha=0.85,
        )

    ax.set_title(model_name.replace("pythia-", ""), fontsize=11)
    ax.set_xlabel("Layer")
    ax.set_xticks(x)
    ax.set_xticklabels([str(l) for l in layers])
    if col == 0:
        ax.set_ylabel("IIA")
    if col == 4:
        ax.legend(fontsize=8)

plt.suptitle(
    "Component Decomposition at Peak Layers (mean across band pairs)", fontsize=12
)
plt.tight_layout()
plt.savefig(
    VIZ_DIR / "T13_02_component_decomposition.png", dpi=150, bbox_inches="tight"
)
plt.close()
print("Saved T13_02_component_decomposition.png")

Saved T13_02_component_decomposition.png


## 13. Viz 3: 5x5 IIA Heatmap

In [13]:
BAND_ORDER = ["low", "medium", "high", "very_high", "control"]
BAND_LABELS = ["L", "M", "H", "VH", "C"]

fig, axes = plt.subplots(1, 5, figsize=(22, 4))

for col, model_name in enumerate(MODELS):
    ax = axes[col]
    df_m = df_matrix[df_matrix["model"] == model_name]

    matrix = np.full((5, 5), np.nan)
    for i, bb in enumerate(BAND_ORDER):
        for j, sb in enumerate(BAND_ORDER):
            vals = df_m[(df_m["base_band"] == bb) & (df_m["source_band"] == sb)]["iia"]
            if len(vals) > 0:
                matrix[i, j] = vals.values[0]

    sns.heatmap(
        matrix,
        ax=ax,
        vmin=0,
        vmax=1,
        cmap="YlOrRd",
        square=True,
        linewidths=0,
        linecolor="none",
        xticklabels=BAND_LABELS,
        yticklabels=BAND_LABELS if col == 0 else False,
        annot=True,
        fmt=".2f",
        annot_kws={"size": 8},
        cbar=col == 4,
    )
    peak_l = peak_layers[model_name][0]
    ax.set_title(f"{model_name.replace('pythia-', '')} (L{peak_l})", fontsize=10)
    if col == 0:
        ax.set_ylabel("base band ->")
    ax.set_xlabel("source band ->")

plt.suptitle(
    "Interchange Intervention Accuracy (IIA) at Peak Layer", fontsize=12, y=1.02
)
plt.tight_layout()
plt.savefig(VIZ_DIR / "T13_03_iia_matrix.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved T13_03_iia_matrix.png")

Saved T13_03_iia_matrix.png


## 14. Viz 4: Position Importance

In [14]:
fig, axes = plt.subplots(1, 5, figsize=(22, 4), sharey=True)

pos_order = list(SWEEP_POSITIONS.keys())

for col, model_name in enumerate(MODELS):
    ax = axes[col]
    df_m = df_pos[df_pos["model"] == model_name]

    # Order by position
    df_m = df_m.set_index("position_name").loc[pos_order].reset_index()

    colors = []
    for pn in df_m["position_name"]:
        if pn.startswith("S"):
            colors.append("#4C72B0")  # source tokens
        elif pn == "T":
            colors.append("#E24A33")  # target
        elif pn.startswith("D"):
            colors.append("#AAAAAA")  # distractors
        else:
            colors.append("#55A868")  # repetition/prediction

    ax.bar(range(len(df_m)), df_m["iia"], color=colors, alpha=0.85)
    ax.set_xticks(range(len(df_m)))
    ax.set_xticklabels(df_m["position_name"], rotation=45, ha="right", fontsize=7)
    ax.set_title(model_name.replace("pythia-", ""), fontsize=10)
    if col == 0:
        ax.set_ylabel("IIA")
    ax.set_xlabel("Position")

plt.suptitle(
    "Position Importance: IIA by Token Position (low->high, peak layer)", fontsize=12
)
plt.tight_layout()
plt.savefig(VIZ_DIR / "T13_04_position_sweep.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved T13_04_position_sweep.png")

Saved T13_04_position_sweep.png


## 15. Cross-Reference with NB05 Edge Importance

In [15]:
# Load edge importance data from NB05 if available
edge_file = ANALYSIS_DIR / "layer_head_sharing.csv"
if edge_file.exists():
    df_edges = pd.read_csv(edge_file)
    print(f"Loaded edge importance: {len(df_edges)} rows")
    print(f"Columns: {df_edges.columns.tolist()}")

    # Correlate: for each model, layer-level edge density vs IIA
    for model_name in MODELS:
        df_m_resid = df_resid[df_resid["model"] == model_name]
        layer_iia = df_m_resid.groupby("layer")["iia"].mean()

        df_m_edges = df_edges[df_edges["model"] == model_name]
        if "layer" in df_m_edges.columns and len(df_m_edges) > 0:
            layer_density = df_m_edges.groupby("layer").size()
            common_layers = sorted(set(layer_iia.index) & set(layer_density.index))
            if len(common_layers) > 3:
                x = [layer_density.get(l, 0) for l in common_layers]
                y = [layer_iia.get(l, 0) for l in common_layers]
                r, p = stats.spearmanr(x, y)
                print(f"{model_name}: edge_density vs IIA: rho={r:.3f}, p={p:.4f}")
else:
    print("NB05 edge importance data not found, skipping cross-reference.")

Loaded edge importance: 410 rows
Columns: ['model', 'layer', 'sharing_level', 'n_edges', 'fraction', 'total_in_layer']
pythia-70m: edge_density vs IIA: rho=nan, p=nan
pythia-160m: edge_density vs IIA: rho=nan, p=nan
pythia-410m: edge_density vs IIA: rho=nan, p=nan
pythia-1b: edge_density vs IIA: rho=nan, p=nan
pythia-1.4b: edge_density vs IIA: rho=nan, p=nan


<TMPDIR>/ipykernel_690905/2927971416.py:20: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, p = stats.spearmanr(x, y)
<TMPDIR>/ipykernel_690905/2927971416.py:20: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, p = stats.spearmanr(x, y)
<TMPDIR>/ipykernel_690905/2927971416.py:20: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, p = stats.spearmanr(x, y)
<TMPDIR>/ipykernel_690905/2927971416.py:20: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, p = stats.spearmanr(x, y)
<TMPDIR>/ipykernel_690905/2927971416.py:20: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, p = stats.spearmanr(x, y)


## 16. Summary

In [16]:
print("=" * 70)
print("ACTIVATION PATCHING SUMMARY")
print("=" * 70)

print("\n--- Peak Layers per Model ---")
for model_name, layers in peak_layers.items():
    df_m = df_resid[df_resid["model"] == model_name]
    overall_iia = df_m.groupby("layer")["iia"].mean()
    print(
        f"{model_name}: L{layers[0]} (IIA={overall_iia[layers[0]]:.3f}), "
        f"L{layers[1]} (IIA={overall_iia[layers[1]]:.3f}), "
        f"L{layers[2]} (IIA={overall_iia[layers[2]]:.3f})"
    )

print("\n--- IIA Matrix Symmetry (same vs cross band) ---")
for model_name in MODELS:
    df_m = df_matrix[df_matrix["model"] == model_name]
    same = df_m[df_m["base_band"] == df_m["source_band"]]["iia"].mean()
    cross = df_m[df_m["base_band"] != df_m["source_band"]]["iia"].mean()
    diff = same - cross
    print(f"{model_name}: same={same:.3f}, cross={cross:.3f}, diff={diff:+.3f}")

print("\n--- Draw Robustness (std across draws) ---")
for model_name in MODELS:
    df_m = df_draws[df_draws["model"] == model_name]
    mean_iia = df_m.groupby("draw")["iia"].mean()
    print(f"{model_name}: mean+/-std = {mean_iia.mean():.3f}+/-{mean_iia.std():.3f}")

print("\n--- Hypothesis Verdicts ---")
overall_cross_iia = df_matrix[df_matrix["base_band"] != df_matrix["source_band"]][
    "iia"
].mean()
print(f"H1 (high IIA across bands): mean cross-band IIA = {overall_cross_iia:.3f}")
if overall_cross_iia > 0.5:
    print("  -> SUPPORTED: Model predictions follow source band at high rates")
else:
    print("  -> INVESTIGATE: Cross-band IIA is low; may indicate band differentiation")

# H2: check if IIA is uniform (low variance across pairs)
pair_iias = df_matrix.groupby(["base_band", "source_band"])["iia"].mean()
cross_pair_iias = pair_iias[
    pair_iias.index.get_level_values(0) != pair_iias.index.get_level_values(1)
]
print(
    f"H2 (uniform IIA): cross-band IIA range = [{cross_pair_iias.min():.3f}, {cross_pair_iias.max():.3f}], "
    f"std = {cross_pair_iias.std():.3f}"
)

print("\nDone.")

ACTIVATION PATCHING SUMMARY

--- Peak Layers per Model ---
pythia-70m: L5 (IIA=0.428), L4 (IIA=0.408), L3 (IIA=0.310)
pythia-160m: L11 (IIA=0.957), L10 (IIA=0.955), L9 (IIA=0.933)
pythia-410m: L23 (IIA=0.985), L22 (IIA=0.980), L21 (IIA=0.978)
pythia-1b: L13 (IIA=0.990), L14 (IIA=0.990), L15 (IIA=0.988)
pythia-1.4b: L21 (IIA=0.973), L19 (IIA=0.973), L20 (IIA=0.968)

--- IIA Matrix Symmetry (same vs cross band) ---
pythia-70m: same=0.466, cross=0.466, diff=+0.000
pythia-160m: same=0.962, cross=0.962, diff=+0.000
pythia-410m: same=0.984, cross=0.984, diff=+0.000
pythia-1b: same=0.986, cross=0.985, diff=+0.001
pythia-1.4b: same=0.968, cross=0.972, diff=-0.004

--- Draw Robustness (std across draws) ---
pythia-70m: mean+/-std = 0.432+/-0.005
pythia-160m: mean+/-std = 0.948+/-0.008
pythia-410m: mean+/-std = 0.985+/-0.000
pythia-1b: mean+/-std = 0.984+/-0.005
pythia-1.4b: mean+/-std = 0.976+/-0.006

--- Hypothesis Verdicts ---
H1 (high IIA across bands): mean cross-band IIA = 0.874
  -> SUPPO